# 01 — Running an experiment

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/balk21/stark-cls-finetune/blob/main/notebooks/01_run_experiment.ipynb)

One experiment = one vot-toolkit run (`vot evaluate`, the *longterm* experiment of VOT-LT2020) on the selected
sequences with one parameter set, plus COCO-style detection metrics and plots.

Flow: **write the parameters → check them → run → look at the results.**
If the setup is not done yet, start with `00_setup.ipynb` (or `00_setup_colab.ipynb` on Google Colab).

The output of every experiment is in `outputs/<experiment name>/`:

| Folder / file | Content |
|---|---|
| `predictions/<seq>/` | `<seq>_001.txt` (boxes), `_confidence.value` (scores), `_time.value` (times), `frames.csv` (all of them + GT + IoU) |
| `tracker_logs/<seq>/` | `finetune_loss.txt` (loss / probabilities of every fine-tuning step), `events.txt` (template / fine-tuning update frames) |
| `plots/<seq>/` | `iou_conf.png` (IoU & score vs. frame), `finetune_loss.png` |
| `metrics/` | `summary.txt`, `metrics.xlsx`, `metrics.json`, `f_curve.csv`, `f_curve.png` |
| `experiment.json` | All parameters, checkpoint, sequence list |
| `run.log` | vot-toolkit output (for debugging) |

In [ ]:
# Run this cell once at the top of every notebook. On Google Colab it also prepares the session
# (clones the repository, restores the environment / dataset / checkpoints from Google Drive).
import sys, subprocess
from pathlib import Path
if "google.colab" in sys.modules:
    REPO = Path("/content/stark-cls-finetune")
    if not REPO.exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/balk21/stark-cls-finetune.git", str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "pull", "-q", "--ff-only"])   # get the latest fixes
    sys.path.insert(0, str(REPO / "notebooks"))
    import nbhelper as nb
    nb.colab_bootstrap()
else:
    sys.path.insert(0, str(Path.cwd()))   # the notebooks/ folder (nbhelper.py lives here)
    import nbhelper as nb

In [ ]:
nb.paths()

## 1. Parameters

Every parameter is explained in detail in the **Parameters** section of the README.
Parameters you do not set take their default values. With `name=None` the experiment name is generated from the
parameters (e.g. `st101_online_pos_lr0.0001_i15_o1_int100_s0`).

Running again with the same name + the same parameters **resumes** (completed sequences are skipped).
The same name + different parameters raises an error (old results are kept); use `run(..., overwrite=True)` to
overwrite. Changing only the `eval_*` parameters does not count as different parameters.

In [ ]:
PARAMS = dict(
    name=None,                    # Output folder name; None = generated automatically

    # --- Model ---
    model="stark_st",             # "stark_st" | "stark_s"
    model_config="baseline_R101", # stark_st: baseline_R101 (ST101), baseline (ST50), *_got10k_only
    checkpoint=None,              # None = official checkpoint; or a file name / path such as "STARKSTcoco_ep0050.pth.tar"

    # --- Data ---
    sequences="all",              # "all" or ["bull", "ballet"]

    # --- Template update ---
    update_interval=100,          # an update is attempted every N frames; 99999 = no updates
    update_conf_thr=0.5,          # the score must be greater than this for an update
    max_template_updates=-1,      # max updates per sequence; -1 = unlimited

    # --- Fine-tuning (stark_st only) ---
    ft_mode="online",             # "none" (plain STARK) | "init" (first frame only) | "online" (first frame + every template update)
    ft_samples="pos",             # "pos" (positive only) | "posneg" (positive + a negative from the same frame)
    ft_lr=1e-4,
    ft_epochs_init=15,            # number of steps on the first frame
    ft_epochs_online=1,           # number of steps in every online fine-tuning session
    ft_weight_decay=1e-4,
    ft_grad_clip_norm=0.1,
    ft_pos_jitter=True,           # apply the ST2 training jitter to the positive box
    ft_center_jitter=4.5,
    ft_scale_jitter=0.5,
    max_ft_updates=-1,            # max online fine-tuning sessions per sequence; -1 = unlimited
    seed=0,

    # --- Evaluation ---
    eval_score_thr=0.35,          # fixed score threshold for the "found" decision (P / R / F1)
    eval_iou_thr=0.5,             # IoU threshold for a correct detection (P / R / F1)
    eval_thr_resolution=100,      # number of candidate thresholds in the F-max search (vot-toolkit: 100)

    # --- vot-toolkit ---
    run_redetection=False,        # the "redetection" experiment of VOT-LT2020 (roughly doubles the run time)
    tracker_timeout=300,          # seconds
)

Check: are the parameters valid, are the checkpoint and the sequences found, where will the output be written?

In [ ]:
info = nb.describe(PARAMS)

## 2. Run

Time: ~30 ms per frame on an RTX 3060 (laptop). All 50 sequences (≈ 215k frames) take **~2 hours**.
For a quick try, run with `sequences=["ballet"]` first (~1 min).

Stopping the cell (■) stops the run. Running again with the same parameters skips the completed sequences.

In [ ]:
OUT = nb.run(PARAMS)
OUT

## 3. Results

**Metrics** (details in the README):
- **F-max threshold** (first line of the summary): 100 candidate thresholds are swept with the VOT-LT method; at every
  threshold P/R per sequence (the "found / not found" counting below) are averaged over sequences and the threshold
  with the highest F is selected. `precision_opt / recall_opt / F_opt` are the values at that threshold. One threshold
  for all sequences.
- `AP / AP50 / AP75`: standard COCO; predictions on frames without the target count as false positives, no score
  threshold is applied.
- `precision / recall / F1`: "found / not found" decision at the fixed `eval_score_thr` and `eval_iou_thr`.
- `absent_reject_rate`: on how many of the frames without the target the tracker said "not found" (score < threshold).
- `legacy_*`: exactly the computation of the old `coco_eval.py` (only for comparison with old results).
- *Mean over sequences*: the mean of each sequence's metric. *Pooled*: all frames as one dataset.

In [ ]:
nb.show_summary(OUT)

In [ ]:
# F-max threshold: P / R / F vs. threshold on the left, precision-recall curve on the right
nb.show_f_curve(OUT)

In [ ]:
# For specific sequences: nb.show_plots(OUT, sequences=["bull", "ballet"])
nb.show_plots(OUT, max_sequences=3)

## 4. Re-analysis with different evaluation settings (tracking is not re-run)

Metrics and plots are regenerated (`metrics/` and `plots/` are overwritten).
`score_thr`: fixed threshold, `iou_thr`: IoU for a correct detection, `thr_resolution`: number of candidates in the
F-max search.

In [ ]:
# nb.analyze(OUT, score_thr=0.5, iou_thr=0.5, thr_resolution=100)
# nb.show_summary(OUT)

## 5. Batch of experiments (grid)

Runs several experiments one after another, starting from `PARAMS` and changing some parameters.
If one experiment fails, the others still run. Set `RUN = True` and run the cell.

In [ ]:
import itertools

GRID = {
    "ft_mode": ["init", "online"],
    "ft_samples": ["pos", "posneg"],
}
param_list = [dict(PARAMS, name=None, **dict(zip(GRID, values))) for values in itertools.product(*GRID.values())]
for p in param_list:
    print({k: p[k] for k in GRID})

RUN = False
if RUN:
    results = nb.run_many(param_list)
    results

To compare experiments, open **`02_compare.ipynb`**.